# 


In [ ]:
import argparse
import json
import logging
import os
import random
import re
import torch
import warnings
import pandas as pd
from typing import Tuple, Iterator, List, Dict
from tqdm.notebook import tqdm
import seaborn as sns


import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SequentialSampler
from sklearn.manifold import TSNE


from os.path import join as pjoin
from collections import defaultdict
from scipy.stats import linregress
from torch.optim import Adam, AdamW

os.environ['PYTHONIOENCODING']='UTF-8'
os.environ['CUDA_LAUNCH_BLOCKING']=str(1)

In [2]:
# Enable automatic reloading of modules before executing code
%load_ext autoreload
%autoreload 2


import plotting as pl
from models import model as md
import utils as ut

In [3]:
# create logger
logger = logging.getLogger('ooo-id-joint')
logger.setLevel(logging.INFO)

# create console handler and set level to debug
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)

# create formatter
formatter = logging.Formatter(
    fmt='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y/%m/%d %H:%M:%S'
)

# add formatter to ch
ch.setFormatter(formatter)

# add ch to logger
logger.addHandler(ch)

In [4]:
task = "odd_one_out"
modality = "behavioral"
triplets_dir = "./data/"
lr = learning_rate = 0.001
lmbda = 0.008
temperature = 1
embed_dim = 5
num_threads = 6
device = "cpu"
batch_size = 100
sampling_method = "normal"
rnd_seed = 42
p = None
results_dir = './results/id-joint/'
plots_dir = './plots/id-joint/'
epochs = 500
distance_metric = "dot"
sparsity = "items_and_random_ids"
lmbda_hierarchical = 100

In [5]:
logger.info("does logging work?")

2025/06/12 16:18:59 - ooo-id-joint - INFO - does logging work?


In [6]:
train_triplets_ID, test_triplets_ID = ut.load_data_ID(device=device, triplets_dir=triplets_dir)
n_items_ID = ut.get_nitems(train_triplets_ID)


...Could not find any .npy files for current modality.
...Now searching for .txt files.



In [7]:
n_participants = len(np.unique(train_triplets_ID.numpy()[:,3]))

In [8]:
#load train and test mini-batches
train_batches, val_batches = ut.load_batches(
    train_triplets=train_triplets_ID,
    test_triplets=test_triplets_ID,
    n_items=n_items_ID,
    batch_size=batch_size,
    sampling_method=sampling_method,
    rnd_seed=rnd_seed,
    p=p, method = "ids"
)

#temperature = torch.tensor(temperature).to(device)
temperature = torch.tensor(temperature).clone().detach()
model = md.SPoSE_ID(
    in_size=n_items_ID, out_size=embed_dim, 
    num_participants=n_participants, init_weights=True)
model.to(device)
optim = Adam(model.parameters(), lr=lr)

In [9]:
# same temperature for everybody, i.e. = 1
temperature = torch.tensor(temperature).clone().detach()
model = md.SPoSE_ID(
    in_size=n_items_ID, out_size=embed_dim, 
    num_participants=n_participants, init_weights=True)
optim = Adam(model.parameters(), lr=lr)

In [10]:
# by-participant dimension weights (random effects) and by-participant softmax scaling
model = md.CombinedModel(
    in_size=n_items_ID, out_size=embed_dim,
    num_participants=n_participants, init_weights=True
)
optim = Adam(model.parameters(), lr=lr)

In [11]:
logger.info(f'...Creating PATHs')

if results_dir == './results/id-joint/':
    results_dir = os.path.join(results_dir, modality, f'{embed_dim}d', str(lmbda), f'seed{rnd_seed:02d}')
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

if plots_dir == './plots/id-joint/':
    plots_dir = os.path.join(plots_dir, modality, f'{embed_dim}d', str(lmbda), f'seed{rnd_seed}')
if not os.path.exists(plots_dir):
    os.makedirs(plots_dir)

model_dir = os.path.join(results_dir, 'model')

2025/06/12 16:19:01 - ooo-id-joint - INFO - ...Creating PATHs


In [12]:
epochs = 2

In [13]:
sparsity = "items_and_random_ids"#"both"#

In [14]:
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
start = 0
train_accs, val_accs = [], []
train_losses, val_losses = [], []
loglikelihoods, complexity_losses_ID, complexity_losses_avg = [], [], []
nneg_d_over_time = []

iter = 0
results = {}
logger.info(f'Optimization started for lambda: {lmbda}\n')

print(f'Optimization started for lambda: {lmbda}\n')
for epoch in tqdm(range(start, epochs)):
    model.train()
    batch_llikelihoods = torch.zeros(len(train_batches))
    batch_closses_avg = torch.zeros(len(train_batches))
    batch_closses_ID = torch.zeros(len(train_batches))
    batch_losses_train = torch.zeros(len(train_batches))
    batch_accs_train = torch.zeros(len(train_batches))
    for i, batch in enumerate(train_batches):
        optim.zero_grad() #zero out gradients
        b = batch[0].to(device)
        id = batch[1].to(device)
        # old code with only embedding model and subsequent fixed softmax function (with temp = 1)
        ##logits = model(b, id)
        #anchor, positive, negative = torch.unbind(torch.reshape(logits, (-1, 3, embed_dim)), dim=1)
        #c_entropy = ut.trinomial_loss(anchor, positive, negative, task, temperature, distance_metric)

        # new code with embedding model and decision models combined (with by-participant tempreatures)
        c_entropy, anchor, positive, negative = model(b, id, distance_metric)
        
        l1_pen_avg = md.l1_regularization(model, "weight").to(device) #L1-norm to enforce sparsity (many 0s)
        l1_pen_ID = md.l1_regularization(model, "individual_slopes", "most").to(device) #L1-norm to enforce sparsity (many 0s)
        W = model.model1.fc.weight
        # positivity constraint to enforce non-negative values in embedding matrix
        pos_pen = torch.sum(F.relu(-W)) + torch.sum(F.relu(-model.model1.individual_slopes.weight))
        temperature = model.model2(id[::3])
        complexity_loss_avg = (lmbda/n_items_ID) * l1_pen_avg
        complexity_loss_ID = (lmbda/n_participants) * l1_pen_ID

        if sparsity == 'items':
            loss = c_entropy + 0.01 * pos_pen + complexity_loss_avg
        elif sparsity == 'both':
            loss = c_entropy + 0.01 * pos_pen + complexity_loss_ID + complexity_loss_avg
        elif sparsity == "items_and_random_ids":
            # Gaussian loss on individual differences for each dimension
            # is only computed by random model
            gaussian_pen = model.model1.hierarchical_loss(id)
            gaussian_loss = gaussian_pen * lmbda_hierarchical
            loss = c_entropy + 0.01 * pos_pen + complexity_loss_avg + gaussian_loss
        
        loss.backward()
        optim.step()
        batch_losses_train[i] += loss.item()
        batch_llikelihoods[i] += c_entropy.item()
        batch_closses_ID[i] += complexity_loss_ID.item()
        batch_closses_avg[i] += complexity_loss_avg.item()
        batch_accs_train[i] += ut.choice_accuracy(anchor, positive, negative, task, distance_metric, scalingfactors=temperature)
        iter += 1

    avg_llikelihood = torch.mean(batch_llikelihoods).item()
    avg_closs_ID = torch.mean(batch_closses_ID).item()
    avg_closs_avg = torch.mean(batch_closses_avg).item()
    avg_train_loss = torch.mean(batch_losses_train).item()
    avg_train_acc = torch.mean(batch_accs_train).item()

    loglikelihoods.append(avg_llikelihood)
    complexity_losses_ID.append(avg_closs_ID)
    complexity_losses_avg.append(avg_closs_avg)
    train_losses.append(avg_train_loss)
    train_accs.append(avg_train_acc)

2025/06/12 16:19:01 - ooo-id-joint - INFO - Optimization started for lambda: 0.008



Optimization started for lambda: 0.008



  0%|          | 0/2 [00:00<?, ?it/s]

In [160]:
scalingfactors = model.model2(id[::3])
#scalingfactors = torch.Tensor([1])
similarities_prep = ut.compute_similarities(anchor, positive, negative, "odd_one_out", distance_metric)
similarities = torch.stack(similarities_prep, dim=-1)
similarities_scaled = similarities/scalingfactors
probas = F.softmax(similarities_scaled, dim=1).detach().cpu().numpy()

In [168]:
similarities[0:5, :]

tensor([[4.0527e-05, 3.8407e-05, 2.1377e-04],
        [2.0684e-01, 7.4430e-03, 3.1893e-03],
        [5.9045e-01, 5.0615e-01, 5.3634e-01],
        [7.3485e-03, 1.5232e-02, 1.7270e-02],
        [1.6996e-05, 2.2824e-03, 1.9777e-03]], grad_fn=<SliceBackward0>)

In [169]:
similarities_scaled[0:5, :]

tensor([[3.1409e-04, 2.9766e-04, 1.6568e-03],
        [1.1275e+00, 4.0571e-02, 1.7385e-02],
        [2.0408e+00, 1.7494e+00, 1.8538e+00],
        [2.2498e-02, 4.6635e-02, 5.2873e-02],
        [5.3763e-05, 7.2201e-03, 6.2562e-03]], grad_fn=<SliceBackward0>)

In [171]:
F.softmax(similarities[0:5, :], dim=1).detach().cpu().numpy()

array([[0.3333143 , 0.33331358, 0.33337206],
       [0.37951186, 0.3109039 , 0.30958426],
       [0.3488584 , 0.32065624, 0.3304854 ],
       [0.3313578 , 0.33398047, 0.33466172],
       [0.3328639 , 0.33361885, 0.3335172 ]], dtype=float32)

In [172]:
F.softmax(similarities_scaled[0:5, :], dim=1).detach().cpu().numpy()

array([[0.33318594, 0.3331805 , 0.3336336 ],
       [0.59996384, 0.2023368 , 0.19769932],
       [0.38809335, 0.29000396, 0.3219027 ],
       [0.32730317, 0.3352992 , 0.33739763],
       [0.33184955, 0.33423623, 0.33391422]], dtype=float32)

In [163]:
similarities/torch.Tensor([0.001])

tensor([[4.0527e-02, 3.8407e-02, 2.1377e-01],
        [2.0684e+02, 7.4430e+00, 3.1893e+00],
        [5.9045e+02, 5.0615e+02, 5.3634e+02],
        [7.3485e+00, 1.5232e+01, 1.7270e+01],
        [1.6996e-02, 2.2824e+00, 1.9777e+00],
        [2.7660e+02, 1.7679e+02, 3.8849e+02],
        [6.8431e-02, 3.0382e+00, 6.0518e+00],
        [2.6287e+02, 3.4189e+02, 1.0957e+02],
        [2.8153e+00, 2.1397e+02, 2.0252e+00],
        [4.7870e+02, 1.2108e+02, 1.1071e+02],
        [4.1374e+02, 3.0084e+02, 2.7412e+02],
        [3.9852e+00, 3.1010e+02, 3.2696e+00],
        [1.3059e+02, 9.7422e+01, 2.1493e+02],
        [2.6581e+02, 5.1948e+02, 1.4032e+02],
        [7.8747e-02, 1.5195e-02, 5.3636e-02],
        [2.6741e+02, 3.1244e+00, 1.7640e+00],
        [3.3571e+02, 9.2376e+01, 1.9539e+02],
        [1.2647e-01, 7.7702e+00, 9.0958e+00],
        [3.4162e+02, 1.7802e+02, 3.3398e+02],
        [5.7694e+02, 1.2888e+02, 1.2611e+02],
        [5.4690e+00, 3.4305e+02, 4.9847e+00],
        [2.7413e+00, 1.7399e+00, 5

In [158]:
np.exp(np.log([.05, .1, .15]))

array([0.05, 0.1 , 0.15])

In [159]:
temperature = 0.0001
temperature = torch.tensor(temperature).clone().detach()
temperature

tensor(1.0000e-04)

In [155]:
probas

array([[0.33318594, 0.3331805 , 0.3336336 ],
       [0.59996384, 0.2023368 , 0.19769932],
       [0.38809335, 0.29000396, 0.3219027 ],
       [0.32730317, 0.3352992 , 0.33739763],
       [0.33184955, 0.33423623, 0.33391422],
       [0.3209559 , 0.24538672, 0.43365735],
       [0.3317436 , 0.333323  , 0.33493343],
       [0.34343997, 0.46865764, 0.18790242],
       [0.3050245 , 0.39023206, 0.30474344],
       [0.71868783, 0.14393555, 0.13737659],
       [0.48618987, 0.27428472, 0.23952544],
       [0.2673139 , 0.46571884, 0.26696718],
       [0.30926827, 0.27540603, 0.41532564],
       [0.20490943, 0.6820652 , 0.11302531],
       [0.33336252, 0.33329973, 0.3333377 ],
       [0.46526855, 0.26774594, 0.26698545],
       [0.6306435 , 0.12329488, 0.24606158],
       [0.3288077 , 0.3350489 , 0.3361433 ],
       [0.39183453, 0.22626182, 0.3819037 ],
       [0.78608626, 0.1076135 , 0.1063002 ],
       [0.17576084, 0.64880735, 0.17543183],
       [0.3278645 , 0.32753524, 0.34460026],
       [0.

In [118]:
probas[0:5, :]

array([[0.3333143 , 0.33331358, 0.33337206],
       [0.37951186, 0.3109039 , 0.30958426],
       [0.3488584 , 0.32065624, 0.3304854 ],
       [0.3313578 , 0.33398047, 0.33466172],
       [0.3328639 , 0.33361885, 0.3335172 ]], dtype=float32)

In [118]:
probas[0:5, :]

array([[0.3333143 , 0.33331358, 0.33337206],
       [0.37951186, 0.3109039 , 0.30958426],
       [0.3488584 , 0.32065624, 0.3304854 ],
       [0.3313578 , 0.33398047, 0.33466172],
       [0.3328639 , 0.33361885, 0.3335172 ]], dtype=float32)

In [105]:
torch.tensor(1.).clone().detach()

tensor(1.)

In [147]:
#scalingfactors = model.model2(id[::3])
scalingfactors = torch.Tensor([.05])
ut.choice_accuracy(anchor, positive, negative, task, distance_metric, scalingfactors=scalingfactors)

np.float64(0.46)

In [33]:
ut.choice_accuracy(anchor, positive, negative, task, distance_metric)

np.float32(nan)

In [56]:
scalingfactors[0:5, :]

tensor([[0.1290],
        [0.1835],
        [0.2893],
        [0.3266],
        [0.3161]], grad_fn=<SliceBackward0>)

In [21]:
train_accs

[0.4570463299751282, 0.48774653673171997]

In [24]:
val_batches

In [25]:
ut.choice_accuracy(anchor, positive, negative, task, distance_metric)

np.float32(0.4592274)

In [28]:
ut.validation(model, val_batches, task, device, level_explanation="ID", modeltype="random_weights_free_scaling")

val_acc =  0.39
val_acc =  0.37
val_acc =  0.36
val_acc =  0.38
val_acc =  0.36
val_acc =  0.36
val_acc =  0.36
val_acc =  0.4
val_acc =  0.42
val_acc =  0.35
val_acc =  0.34
val_acc =  0.34
val_acc =  0.42
val_acc =  0.48
val_acc =  0.47
val_acc =  0.47
val_acc =  0.42
val_acc =  0.47
val_acc =  0.44
val_acc =  0.45
val_acc =  0.48
val_acc =  0.48
val_acc =  0.43
val_acc =  0.48
val_acc =  0.45
val_acc =  0.44
val_acc =  0.44
val_acc =  0.45
val_acc =  0.35
val_acc =  0.44
val_acc =  0.37
val_acc =  0.35
val_acc =  0.34
val_acc =  0.34
val_acc =  0.34
val_acc =  0.35
val_acc =  0.34
val_acc =  0.34
val_acc =  0.51
val_acc =  0.47
val_acc =  0.47
val_acc =  0.47
val_acc =  0.5
val_acc =  0.55
val_acc =  0.52
val_acc =  0.51
val_acc =  0.52
val_acc =  0.5
val_acc =  0.47
val_acc =  0.46
val_acc =  0.48
val_acc =  0.49
val_acc =  0.48
val_acc =  0.45
val_acc =  0.43
val_acc =  0.4
val_acc =  0.45
val_acc =  0.39
val_acc =  0.37
val_acc =  0.4
val_acc =  0.36
val_acc =  0.36
val_acc =  0.

(0.9751155972480774, 0.4167383313179016)

In [ ]:
sims[2][0]

In [ ]:
torch.mean(-torch.log(F.softmax(torch.stack(sims, dim=-1), dim=1)[:, 0]))

In [ ]:
logits2 = torch.stack(sims, dim=-1)

In [ ]:
logits2.shape

In [ ]:
ts_by_id.shape

In [ ]:
logits2_scaled.shape

In [ ]:
ts_by_id = torch.tensor(np.concatenate((np.ones(25), np.ones(50)*2, np.ones(25)*3)))
ts_by_id = ts_by_id.unsqueeze(1)
logits2_scaled = logits2/ts_by_id
probs = F.softmax(logits2_scaled, dim = 1)

#todos
- create an embedding similar as or individual betas
- extract relevant embeddings for in-batch ids
- divide sims by by-id scalings
- apply normal softmax

In [ ]:
probs

In [ ]:
np.c

In [ ]:
np.concatenate(np.ones(25), np.ones(50)*2)

In [ ]:
model.global_mean

In [ ]:
model.global_std

In [ ]:
betas = model.individual_slopes.weight.detach().numpy()

In [ ]:
# Create histograms for each column
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

for i in range(5):
    axes[i].hist(betas[:, i], bins=20, alpha=0.7)
    axes[i].set_title(f'Histogram {i+1}')

plt.tight_layout()
plt.show()

In [ ]:
for n, p in model.named_parameters():
    print(n)
    print(p)

In [ ]:
kwd_pattern = fr'{"weight"}'
agreement = "few"
l1_reg = torch.tensor(0., requires_grad=True)
for n, p in model.named_parameters():
    if re.search(kwd_pattern, n):
        if agreement == "few":
            l1_reg = l1_reg + torch.norm(p, 1)
        elif agreement == "most":
            l1_reg = l1_reg + torch.norm(1-p, 1)
        l1_reg = l1_reg + torch.norm(p, 1)


In [ ]:
p.shape

In [ ]:
l1_reg

In [ ]:
torch.sum(F.relu(-W)) + torch.sum(F.relu(-model.individual_slopes.weight))

In [ ]:
F.relu(-W)

In [ ]:
md.l1_regularization(model, "individual_slopes", agreement="few").to(device)

In [ ]:
torch.sum(F.relu(-model.individual_slopes.weight))

In [ ]:
model.individual_slopes.weight.shape

In [ ]:
torch.sum(F.relu(-W))

In [ ]:
plt.hist(pd.melt(pd.DataFrame(W.detach().numpy()))["value"])

In [ ]:
plt.hist(pd.melt(pd.DataFrame(model.individual_slopes.weight.detach().numpy()))["value"])

In [ ]:
torch.norm(model.named_parameters

In [ ]:
sns.displot(model.individual_slopes(torch.LongTensor([0])).detach().numpy().T, binwidth=1)

In [ ]:
ut.validation(model, val_batches, task, device, level_explanation="ID")

In [ ]:
anchor = torch.Tensor([[3, 10]])
positive = torch.Tensor([[1, 3]])
negative = torch.Tensor([[7, 5]])

In [ ]:
ut.choice_accuracy(anchor, positive, negative, method="odd_one_out")

In [ ]:
model.individual_slopes(torch.LongTensor(1))